# 07 - Cleaning the Quality Assurance data

This notebook cleans and prepares the data that goes beyond the shop
floor: customers and sales, customer complaints, suppliers, incoming
raw material inspection, supplier complaints, and nonconformance/CAPA.
Same cleaning discipline as the previous notebooks.


In [ ]:
import pandas as pd
import numpy as np
import etl_lib as etl

RAW = "../../datasets/raw"
DIM = "../../datasets/dim"
PROCESSED = "../../datasets/processed"


## Step 1: load and clean

In [ ]:
def clean(df, category_columns, name):
    df = etl.clean_disguised_blanks(df)
    df = etl.strip_extra_spaces(df)
    df = etl.standardize_categories(df, category_columns)
    df, removed_count = etl.drop_duplicate_rows(df)
    print(f"[{name}] {removed_count} duplicates removed -> {len(df):,} rows")
    return df


dim_customer = pd.read_csv(f"{DIM}/dim_customer.csv", encoding="utf-8-sig")
dim_supplier = pd.read_csv(f"{DIM}/dim_supplier.csv", encoding="utf-8-sig")
dim_raw_material_control_plan = pd.read_csv(f"{DIM}/dim_raw_material_control_plan.csv", encoding="utf-8-sig")

sales = clean(pd.read_csv(f"{RAW}/fact_sales_raw.csv", encoding="utf-8-sig"),
              ["ProductFamily", "Process"], "sales")
complaints = clean(pd.read_csv(f"{RAW}/fact_customer_complaints_raw.csv", encoding="utf-8-sig"),
                    ["DefectType", "Severity", "Status"], "customer complaints")
raw_material_inspection = clean(pd.read_csv(f"{RAW}/fact_raw_material_inspection_raw.csv", encoding="utf-8-sig"),
                                 ["Material", "Result"], "raw material inspection")
raw_material_disposition = clean(pd.read_csv(f"{RAW}/fact_raw_material_lot_disposition_raw.csv", encoding="utf-8-sig"),
                                  ["Material", "FinalDecision"], "raw material lot disposition")
supplier_complaints = clean(pd.read_csv(f"{RAW}/fact_supplier_complaints_raw.csv", encoding="utf-8-sig"),
                             ["IssueType", "Status"], "supplier complaints")
nonconformance = clean(pd.read_csv(f"{RAW}/fact_nonconformance_raw.csv", encoding="utf-8-sig"),
                        ["Type", "Area", "Severity"], "nonconformance")
capa = clean(pd.read_csv(f"{RAW}/fact_capa_raw.csv", encoding="utf-8-sig"),
             ["Status", "CAPAType", "Severity"], "CAPA")


## Step 2: calendar columns

The same ISO week convention used everywhere else in the project,
applied to every table with a date -- keeps monthly/weekly trend charts
consistent across domains.


In [ ]:
date_tables = (sales, complaints, raw_material_inspection, raw_material_disposition,
               supplier_complaints, nonconformance)

for table in date_tables:
    calendar_columns = etl.add_calendar_columns(table, "Date")
    table["ISOWeek"] = calendar_columns["ISOWeek"]
    table["ISOWeekday"] = calendar_columns["ISOWeekday"]
    table["Month"] = pd.to_datetime(table["Date"]).dt.to_period("M").astype(str)

capa["OpenMonth"] = pd.to_datetime(capa["OpenDate"]).dt.to_period("M").astype(str)

print("Calendar columns added.")


## Step 3: customer complaints -- resolution time

In [ ]:
complaints = etl.compute_days_between(complaints, "Date", "ResolutionDate", "ResolutionDays")
complaints[["ComplaintId", "Status", "Date", "ResolutionDate", "ResolutionDays"]].head()


## Step 4: supplier complaints -- response and resolution time

This is where **average supplier response time** comes from:
`ResponseDays` counts the days between us reporting the issue and the
supplier's first response -- kept separate from total resolution time
(`ResolutionDays`), since "responded quickly" and "actually fixed it"
are two different things.


In [ ]:
supplier_complaints = etl.compute_days_between(supplier_complaints, "Date", "DateSupplierResponded", "ResponseDays")
supplier_complaints = etl.compute_days_between(supplier_complaints, "Date", "DateResolved", "ResolutionDays")
supplier_complaints[["SupplierComplaintId", "SupplierId", "Status", "ResponseDays", "ResolutionDays"]].head()


## Step 5: CAPA -- closure time and overdue flag

`ClosureDays` only makes sense for CAPAs that already closed (it stays
blank for the rest -- an open CAPA didn't take "0 days", it simply
hasn't finished yet). `IsOverdue` is recomputed here from `DueDate`.


In [ ]:
capa = etl.compute_days_between(capa, "OpenDate", "CloseDate", "ClosureDays")
capa["IsOverdue"] = capa["Status"] == "Overdue"
capa[["CAPAId", "Status", "OpenDate", "DueDate", "CloseDate", "ClosureDays", "IsOverdue"]].head()


## Step 6: raw material -- acceptance flag

In [ ]:
raw_material_disposition["IsAccepted"] = raw_material_disposition["FinalDecision"] == "Accepted"
raw_material_disposition["IsRejected"] = raw_material_disposition["FinalDecision"] == "Rejected"
raw_material_disposition[["MaterialLotId", "SupplierId", "FinalDecision", "IsAccepted", "IsRejected"]].head()


## Step 7: save the processed tables

In [ ]:
dim_customer.to_csv(f"{PROCESSED}/dim_customer.csv", index=False, encoding="utf-8-sig")
dim_supplier.to_csv(f"{PROCESSED}/dim_supplier.csv", index=False, encoding="utf-8-sig")
dim_raw_material_control_plan.to_csv(f"{PROCESSED}/dim_raw_material_control_plan.csv", index=False, encoding="utf-8-sig")

sales.to_csv(f"{PROCESSED}/fact_sales_processed.csv", index=False, encoding="utf-8-sig")
complaints.to_csv(f"{PROCESSED}/fact_customer_complaints_processed.csv", index=False, encoding="utf-8-sig")
raw_material_inspection.to_csv(f"{PROCESSED}/fact_raw_material_inspection_processed.csv", index=False, encoding="utf-8-sig")
raw_material_disposition.to_csv(f"{PROCESSED}/fact_raw_material_lot_disposition_processed.csv", index=False, encoding="utf-8-sig")
supplier_complaints.to_csv(f"{PROCESSED}/fact_supplier_complaints_processed.csv", index=False, encoding="utf-8-sig")
nonconformance.to_csv(f"{PROCESSED}/fact_nonconformance_processed.csv", index=False, encoding="utf-8-sig")
capa.to_csv(f"{PROCESSED}/fact_capa_processed.csv", index=False, encoding="utf-8-sig")

print("Quality Assurance tables saved:")
saved_tables = {
    "sales": sales, "complaints": complaints, "raw_material_inspection": raw_material_inspection,
    "raw_material_disposition": raw_material_disposition, "supplier_complaints": supplier_complaints,
    "nonconformance": nonconformance, "capa": capa,
}
for name, table in saved_tables.items():
    print(f"  {name}: {table.shape}")
